In [ ]:
"""
Food Inspection Metrics — CdTe on Medipix3
Computes: CNR, SNR, Pd/FPR/ROC/AUC, material separability (Bhattacharyya),
and analytical count-rate dead-time loss curves.

METHODOLOGY NOTES (read before presenting to Alex):
- Contaminant attenuation calculated analytically via NIST mass attenuation
  coefficients for iron (steel proxy), not simulated as an object in Allpix².
- Pd/FPR uses a Poisson statistical model parameterised by simulated mean
  hit counts, scaled to an assumed contaminant footprint (1% of image area
  by default) — this is a detectability MODEL, not a real detection algorithm.
- Dead-time model uses Medipix3's published "time to peak" (120ns) as a
  proxy for per-pixel dead time — real chip behaviour depends on counter
  architecture (configurable 1-24 bit counters) which this simplifies.
"""

import os
os.environ["QT_QPA_PLATFORM"] = "xcb"

import uproot
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import poisson
from scipy.integrate import trapezoid

BASE = os.path.expanduser("~/itek/senstech-simulation/data")
DET = "medipix3_detector"

ENERGIES = [40, 60, 80]
THICKNESSES = ["0p5mm", "1mm", "2mm"]
THICK_LABELS = {"0p5mm": "0.5mm", "1mm": "1mm", "2mm": "2mm"}
N_BASELINE = 10000
ROI_FRACTION = 0.01  # assumed contaminant footprint as fraction of image — ADJUST per real fragment size

# ── Load helper ───────────────────────────────────────────────────────────
def load_hits_and_spectrum(path):
    f = uproot.open(path)
    pc = f[f"DetectorHistogrammer/{DET}/charge/pixel_charge;1"].to_numpy()
    cc = f[f"DetectorHistogrammer/{DET}/charge/cluster_charge;1"].to_numpy()
    hits = int(pc[0].sum())
    return hits, pc, cc

# ── Bhattacharyya distance ─────────────────────────────────────────────────
def bhattacharyya(hist1, hist2):
    p = hist1 / (hist1.sum() + 1e-12)
    q = hist2 / (hist2.sum() + 1e-12)
    bc = np.sum(np.sqrt(p * q))
    bc = min(bc, 1.0)
    return -np.log(bc + 1e-12)

# ── Load all data ─────────────────────────────────────────────────────────
print("Loading background (clean) files...")
background = {}
for E in ENERGIES:
    path = os.path.join(BASE, f"s1_cdte_{E}keV.root")
    hits, pc, cc = load_hits_and_spectrum(path)
    background[E] = {"hits": hits, "pixel_charge": pc, "cluster_charge": cc}
    print(f"  {E} keV background: {hits:,} hits")

print("\nLoading contaminant files...")
contaminant = {}
for E in ENERGIES:
    for T in THICKNESSES:
        path = os.path.join(BASE, f"food_contam_{E}keV_{T}.root")
        if not os.path.exists(path):
            print(f"  MISSING: food_contam_{E}keV_{T}.root")
            continue
        hits, pc, cc = load_hits_and_spectrum(path)
        contaminant[(E, T)] = {"hits": hits, "pixel_charge": pc, "cluster_charge": cc}
        print(f"  {E} keV / {T}: {hits:,} hits")

# ── Compute CNR, SNR, Bhattacharyya for every combination ─────────────────
results = []
for E in ENERGIES:
    bg_hits = background[E]["hits"]
    bg_cc = background[E]["cluster_charge"]
    for T in THICKNESSES:
        if (E, T) not in contaminant:
            continue
        c_hits = contaminant[(E, T)]["hits"]
        c_cc = contaminant[(E, T)]["cluster_charge"]

        # CNR: pooled Poisson noise model
        sigma_bg = np.sqrt(max(bg_hits, 1))
        sigma_c = np.sqrt(max(c_hits, 1))
        cnr = abs(bg_hits - c_hits) / np.sqrt((sigma_bg**2 + sigma_c**2) / 2)

        snr_bg = bg_hits / sigma_bg
        snr_c = c_hits / sigma_c

        # Material separability — Bhattacharyya distance between cluster charge spectra
        bd = bhattacharyya(bg_cc[0], c_cc[0])

        # Pd / FPR via Poisson model scaled to assumed contaminant ROI
        mean_bg_roi = bg_hits * ROI_FRACTION
        mean_c_roi = c_hits * ROI_FRACTION

        # Sweep threshold, compute Pd (flag if count < threshold) and FPR
        thresholds = np.linspace(0, mean_bg_roi * 1.5, 200)
        pd_curve = poisson.cdf(thresholds, mean_c_roi)   # P(count < T | contaminant)
        fpr_curve = poisson.cdf(thresholds, mean_bg_roi) # P(count < T | clean) = false alarm

        # AUC via trapezoidal rule on ROC (fpr vs pd)
        auc = trapezoid(pd_curve[order], fpr_curve[order])

        # Operating point: threshold at 5% FPR
        idx_5pct = np.argmin(np.abs(fpr_curve - 0.05))
        pd_at_5pct_fpr = pd_curve[idx_5pct]

        results.append({
            "energy": E, "thickness": T, "thickness_label": THICK_LABELS[T],
            "bg_hits": bg_hits, "c_hits": c_hits,
            "transmission_pct": (c_hits / bg_hits) * 100 if bg_hits else 0,
            "cnr": cnr, "snr_bg": snr_bg, "snr_c": snr_c,
            "bhattacharyya": bd, "auc": auc, "pd_at_5pct_fpr": pd_at_5pct_fpr,
            "fpr_curve": fpr_curve, "pd_curve": pd_curve,
        })

# ── Print summary table ────────────────────────────────────────────────────
print("\n" + "═"*90)
print("FOOD INSPECTION METRICS SUMMARY")
print("═"*90)
print(f"{'Energy':<8}{'Thick':<8}{'Transmission':<14}{'CNR':<8}{'SNR(bg)':<10}{'Bhatt.':<10}{'AUC':<8}{'Pd@5%FPR':<10}")
for r in results:
    print(f"{r['energy']:<8}{r['thickness_label']:<8}{r['transmission_pct']:<13.1f}%"
          f"{r['cnr']:<8.2f}{r['snr_bg']:<10.1f}{r['bhattacharyya']:<10.3f}"
          f"{r['auc']:<8.3f}{r['pd_at_5pct_fpr']:<10.3f}")
print("═"*90)

# ── Analytical count-rate dead-time model ──────────────────────────────────
TAU = 120e-9  # seconds, Medipix3 time-to-peak used as dead-time proxy (caveat above)
true_rates = np.logspace(2, 8, 200)  # counts/s/pixel, swept range
m_nonparalyzable = true_rates / (1 + true_rates * TAU)
m_paralyzable = true_rates * np.exp(-true_rates * TAU)
loss_nonparalyzable = 1 - (m_nonparalyzable / true_rates)
loss_paralyzable = 1 - (m_paralyzable / true_rates)

# Datasheet reference: 826 Mcounts/mm²/s -> per-pixel (55um x 55um = 3.025e-5 mm²)
pixel_area_mm2 = 0.055 * 0.055
max_rate_per_pixel = 826e6 * pixel_area_mm2  # counts/s/pixel

# ── Visualisation ───────────────────────────────────────────────────────────
NAVY, WHITE, LGRAY, GRAY = "#0B1F3A", "#F8FAFC", "#CBD5E1", "#64748B"
TEAL, BLUE, PURP, AMBER, RED, GREEN = "#0D9488", "#0369A1", "#7C3AED", "#D97706", "#DC2626", "#16A34A"

def style_ax(ax, title):
    ax.set_facecolor("#0D2040")
    ax.tick_params(colors=LGRAY, labelsize=9)
    ax.xaxis.label.set_color(LGRAY); ax.yaxis.label.set_color(LGRAY)
    ax.title.set_color(WHITE)
    ax.set_title(title, fontweight="bold", fontsize=11, pad=10)
    for s in ax.spines.values(): s.set_edgecolor("#1E3A5F")
    ax.grid(True, alpha=0.15, color=LGRAY)

fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor(NAVY)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.35,
                       left=0.06, right=0.97, top=0.93, bottom=0.05)

energy_colors = {40: TEAL, 60: BLUE, 80: PURP}

# Panel 1 — CNR vs thickness, grouped by energy
ax1 = fig.add_subplot(gs[0, 0])
style_ax(ax1, "CNR vs Contaminant Thickness")
for E in ENERGIES:
    subset = [r for r in results if r["energy"] == E]
    x = [THICKNESSES.index(r["thickness"]) for r in subset]
    y = [r["cnr"] for r in subset]
    ax1.plot(x, y, marker="o", markersize=9, linewidth=2.5,
             color=energy_colors[E], label=f"{E} keV", markerfacecolor=WHITE)
ax1.set_xticks(range(len(THICKNESSES)))
ax1.set_xticklabels([THICK_LABELS[t] for t in THICKNESSES])
ax1.set_xlabel("Steel Fragment Thickness")
ax1.set_ylabel("CNR")
ax1.axhline(y=3, color=AMBER, linestyle="--", linewidth=1.5,
            label="Typical detectability (CNR=3)")
ax1.legend(fontsize=9, labelcolor=WHITE, facecolor="#0D2040", edgecolor="#1E3A5F")

# Panel 2 — Transmission fraction (the physics input)
ax2 = fig.add_subplot(gs[0, 1])
style_ax(ax2, "Steel Transmission Fraction (NIST-derived)")
for E in ENERGIES:
    subset = [r for r in results if r["energy"] == E]
    x = [THICKNESSES.index(r["thickness"]) for r in subset]
    y = [r["transmission_pct"] for r in subset]
    ax2.plot(x, y, marker="s", markersize=9, linewidth=2.5,
             color=energy_colors[E], label=f"{E} keV", markerfacecolor=WHITE)
ax2.set_xticks(range(len(THICKNESSES)))
ax2.set_xticklabels([THICK_LABELS[t] for t in THICKNESSES])
ax2.set_xlabel("Steel Fragment Thickness")
ax2.set_ylabel("Transmission (%)")
ax2.legend(fontsize=9, labelcolor=WHITE, facecolor="#0D2040", edgecolor="#1E3A5F")

# Panel 3 — Bhattacharyya separability
ax3 = fig.add_subplot(gs[0, 2])
style_ax(ax3, "Material Separability (Bhattacharyya Distance)")
for E in ENERGIES:
    subset = [r for r in results if r["energy"] == E]
    x = [THICKNESSES.index(r["thickness"]) for r in subset]
    y = [r["bhattacharyya"] for r in subset]
    ax3.plot(x, y, marker="^", markersize=9, linewidth=2.5,
             color=energy_colors[E], label=f"{E} keV", markerfacecolor=WHITE)
ax3.set_xticks(range(len(THICKNESSES)))
ax3.set_xticklabels([THICK_LABELS[t] for t in THICKNESSES])
ax3.set_xlabel("Steel Fragment Thickness")
ax3.set_ylabel("Bhattacharyya Distance (higher = more separable)")
ax3.legend(fontsize=9, labelcolor=WHITE, facecolor="#0D2040", edgecolor="#1E3A5F")

# Panel 4-6 — ROC curves for 60 keV (most relevant energy) at each thickness
for i, T in enumerate(THICKNESSES):
    ax = fig.add_subplot(gs[1, i])
    style_ax(ax, f"ROC — 60 keV, {THICK_LABELS[T]} steel")
    r = next((r for r in results if r["energy"] == 60 and r["thickness"] == T), None)
    if r:
        ax.plot(r["fpr_curve"], r["pd_curve"], color=TEAL, linewidth=2.5)
        ax.plot([0, 1], [0, 1], color=GRAY, linestyle="--", linewidth=1, alpha=0.6)
        ax.fill_between(r["fpr_curve"], r["pd_curve"], alpha=0.15, color=TEAL)
        ax.text(0.6, 0.15, f"AUC = {r['auc']:.3f}", color=WHITE,
                fontsize=11, fontweight="bold",
                bbox=dict(boxstyle="round", facecolor="#0D2040", edgecolor=TEAL))
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("Probability of Detection")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# Panel 7 — Dead-time loss curves (analytical, no sim needed)
ax7 = fig.add_subplot(gs[2, 0:2])
style_ax(ax7, "Count-Rate Dead-Time Loss (Analytical Model)")
ax7.plot(true_rates, loss_nonparalyzable * 100, color=TEAL, linewidth=2.5,
         label="Non-paralyzable model")
ax7.plot(true_rates, loss_paralyzable * 100, color=PURP, linewidth=2.5,
         label="Paralyzable model")
ax7.axvline(x=max_rate_per_pixel, color=AMBER, linestyle="--", linewidth=2,
            label=f"Datasheet max ({max_rate_per_pixel/1000:.0f} kHz/pixel)")
ax7.set_xscale("log")
ax7.set_xlabel("True Photon Rate (counts/s/pixel)")
ax7.set_ylabel("Count Loss (%)")
ax7.set_ylim(0, 100)
ax7.legend(fontsize=9, labelcolor=WHITE, facecolor="#0D2040", edgecolor="#1E3A5F")

# Panel 8 — Summary / methodology notes
ax8 = fig.add_subplot(gs[2, 2])
ax8.set_facecolor("#0D2040")
ax8.axis("off")
style_ax(ax8, "Methodology Notes")
notes = [
    "Contaminant: steel, NIST attenuation",
    "Detection model: Poisson, 1% ROI",
    "Dead-time τ: 120ns (time-to-peak proxy)",
    "Background: existing S1 energy sweep",
    "All values are model estimates —",
    "validate against real measurements",
]
for i, n in enumerate(notes):
    ax8.text(0.02, 0.9 - i*0.15, n, color=LGRAY, fontsize=10,
             transform=ax8.transAxes, va="top")

fig.suptitle("Food Inspection Detectability Analysis — Steel Contaminant in CdTe/Medipix3\n"
             "CNR, ROC/AUC, Material Separability & Count-Rate Limits  ·  University of Surrey × Sens Tech",
             fontsize=14, fontweight="bold", color=WHITE, y=0.98)

out_dir = os.path.expanduser("~/itek/senstech-simulation/analysis/notebooks")
os.makedirs(out_dir, exist_ok=True)
out = os.path.join(out_dir, "food_inspection_dashboard.png")
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"\nSaved: {out}")

Loading background (clean) files...
  40 keV background: 16,547 hits
  60 keV background: 12,554 hits
  80 keV background: 7,786 hits

Loading contaminant files...
  40 keV / 0p5mm: 3,982 hits
  40 keV / 1mm: 966 hits
  40 keV / 2mm: 50 hits
  60 keV / 0p5mm: 7,768 hits
  60 keV / 1mm: 4,786 hits
  60 keV / 2mm: 1,865 hits
  80 keV / 0p5mm: 6,140 hits
  80 keV / 1mm: 4,865 hits
  80 keV / 2mm: 3,044 hits


AttributeError: module 'numpy' has no attribute 'trapz'